In [2]:
import pandas as pd
import numpy as np
import re
import os
import nltk

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer



[nltk_data] Downloading package stopwords to
[nltk_data]     /home/mohamed/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/mohamed/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/mohamed/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
df = pd.read_csv("../data/Reviews.csv")
print(f"Shape: {df.shape}")

# Créer le label sentiment
def score_to_label(s):
    if s < 3:  return 0   # negative
    if s == 3: return 1   # neutral
    return 2              # positive

df['sentiment'] = df['Score'].apply(score_to_label)
print("Labels créés ")
print(df['sentiment'].value_counts().sort_index())

Shape: (568454, 10)
Labels créés 
sentiment
0     82037
1     42640
2    443777
Name: count, dtype: int64


In [4]:
# Stopwords SANS les mots de négation
stop_words = set(stopwords.words('english'))
negations = {'no', 'not', 'nor', 'never', 'neither', 'none',
             "don't", "doesn't", "didn't", "won't", "wouldn't",
             "can't", "cannot", "isn't", "aren't", "wasn't", "weren't"}
stop_words = stop_words - negations  # retirer les négations

lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if pd.isna(text) or str(text).strip() == '':
        return ""
    text = str(text).lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t)
              for t in tokens
              if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

# Test rapide
print(clean_text("This is <br/> GREAT product!!! ??<h1>http://amazon.com 👍"))
print(clean_text("Not as advertised,w22112 very disappointed..."))

great product
not advertised disappointed


In [5]:
print("Création full_text...")
df['full_text'] = df['Summary'].fillna('') + '. ' + df['Text'].fillna('')

print("Nettoyage du texte (patience ~5-8 min)...")
df['cleaned'] = df['full_text'].apply(clean_text)

# Vérifier
print(f"\n✅ Cleaning terminé!")
print(f"Exemples cleaned nulls: {(df['cleaned'] == '').sum()}")
print("\n--- Exemple 1 ---")
print(f"Original: {df['full_text'].iloc[1][:100]}")
print(f"Cleaned:  {df['cleaned'].iloc[1][:100]}")
print("\n--- Exemple 2 ---")
print(f"Original: {df['full_text'].iloc[13][:100]}")
print(f"Cleaned:  {df['cleaned'].iloc[13][:100]}")

Création full_text...
Nettoyage du texte (patience ~5-8 min)...

✅ Cleaning terminé!
Exemples cleaned nulls: 2

--- Exemple 1 ---
Original: Not as Advertised. Product arrived labeled as Jumbo Salted Peanuts...the peanuts were actually small
Cleaned:  not advertised product arrived labeled jumbo salted peanut peanut actually small sized unsalted not 

--- Exemple 2 ---
Original: fresh and greasy!. good flavor! these came securely packed... they were fresh and delicious! i love 
Cleaned:  fresh greasy good flavor came securely packed fresh delicious love twizzlers


In [6]:
from sklearn.model_selection import train_test_split

# Garder seulement les colonnes utiles
df_clean = df[['Id', 'ProductId', 'UserId', 'Time', 
               'Summary', 'Text', 'cleaned', 
               'Score', 'sentiment']].copy()

# Split stratifié (garde les proportions de classes)
train_df, temp_df = train_test_split(
    df_clean, test_size=0.2, 
    stratify=df_clean['sentiment'], 
    random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, 
    stratify=temp_df['sentiment'], 
    random_state=42
)

print(f"✅ Split terminé:")
print(f"  Train: {len(train_df):,}  ({len(train_df)/len(df_clean)*100:.0f}%)")
print(f"  Val:   {len(val_df):,}   ({len(val_df)/len(df_clean)*100:.0f}%)")
print(f"  Test:  {len(test_df):,}   ({len(test_df)/len(df_clean)*100:.0f}%)")

print(f"\nDistribution dans train:")
print(train_df['sentiment'].value_counts().sort_index())

✅ Split terminé:
  Train: 454,763  (80%)
  Val:   56,845   (10%)
  Test:  56,846   (10%)

Distribution dans train:
sentiment
0     65630
1     34112
2    355021
Name: count, dtype: int64


In [7]:
os.makedirs("../data", exist_ok=True)

# Quoting explicite pour éviter les problèmes de virgules
import csv

print("Sauvegarde train.csv...")
train_df.to_csv("../data/train.csv", index=False, quoting=csv.QUOTE_ALL)

print("Sauvegarde val.csv...")
val_df.to_csv("../data/val.csv", index=False, quoting=csv.QUOTE_ALL)

print("Sauvegarde test_set.csv...")
test_df.to_csv("../data/test_set.csv", index=False, quoting=csv.QUOTE_ALL)

# Vérification
for f in ["train.csv", "val.csv", "test_set.csv"]:
    size = os.path.getsize(f"../data/{f}") / (1024*1024)
    # Vérifier nulls
    check = pd.read_csv(f"../data/{f}")
    nulls = check['sentiment'].isna().sum()
    print(f"  {f}: {size:.1f} MB | sentiment nulls: {nulls}")

print("\n✅ Sauvegardé avec quoting correct")

Sauvegarde train.csv...


Sauvegarde val.csv...
Sauvegarde test_set.csv...
  train.csv: 348.8 MB | sentiment nulls: 0
  val.csv: 43.3 MB | sentiment nulls: 0
  test_set.csv: 43.6 MB | sentiment nulls: 0

✅ Sauvegardé avec quoting correct


In [8]:
import json

stats = {
    "total": len(df_clean),
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df),
    "classes": {
        "negative": int((df_clean['sentiment']==0).sum()),
        "neutral":  int((df_clean['sentiment']==1).sum()),
        "positive": int((df_clean['sentiment']==2).sum()),
    },
    "avg_cleaned_len": float(df_clean['cleaned'].str.len().mean())
}

with open("../docs/preprocessing_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print("📊 Stats finales:")
print(json.dumps(stats, indent=2))
print("\n✅ Notebook 02 terminé! Prêt pour le training.")

📊 Stats finales:
{
  "total": 568454,
  "train": 454763,
  "val": 56845,
  "test": 56846,
  "classes": {
    "negative": 82037,
    "neutral": 42640,
    "positive": 443777
  },
  "avg_cleaned_len": 275.00814313911064
}

✅ Notebook 02 terminé! Prêt pour le training.


In [9]:
import csv

# Garder SEULEMENT les colonnes nécessaires
cols_needed = ['Id', 'ProductId', 'UserId', 'Time', 
               'Summary', 'Text', 'cleaned', 'sentiment']

train_clean = train_df[cols_needed].copy()
val_clean   = val_df[cols_needed].copy()
test_clean  = test_df[cols_needed].copy()

# Vérifier avant sauvegarde
print("Vérification avant sauvegarde:")
print(f"  train sentiment unique: {sorted(train_clean['sentiment'].unique())}")
print(f"  train sentiment nulls: {train_clean['sentiment'].isna().sum()}")

# Sauvegarder avec quoting strict
train_clean.to_csv("../data/train.csv", index=False, quoting=csv.QUOTE_ALL)
val_clean.to_csv("../data/val.csv",     index=False, quoting=csv.QUOTE_ALL)
test_clean.to_csv("../data/test_set.csv", index=False, quoting=csv.QUOTE_ALL)

# Vérifier après sauvegarde
for f in ["train.csv", "val.csv", "test_set.csv"]:
    check = pd.read_csv(f"../data/{f}")
    print(f"\n{f}:")
    print(f"  Colonnes: {list(check.columns)}")
    print(f"  sentiment unique: {sorted(check['sentiment'].unique())}")
    print(f"  sentiment nulls: {check['sentiment'].isna().sum()}")
    size = os.path.getsize(f"../data/{f}") / (1024*1024)
    print(f"  Size: {size:.1f} MB")

print("\n✅ Done!")

Vérification avant sauvegarde:
  train sentiment unique: [np.int64(0), np.int64(1), np.int64(2)]
  train sentiment nulls: 0

train.csv:
  Colonnes: ['Id', 'ProductId', 'UserId', 'Time', 'Summary', 'Text', 'cleaned', 'sentiment']
  sentiment unique: [np.int64(0), np.int64(1), np.int64(2)]
  sentiment nulls: 0
  Size: 347.0 MB

val.csv:
  Colonnes: ['Id', 'ProductId', 'UserId', 'Time', 'Summary', 'Text', 'cleaned', 'sentiment']
  sentiment unique: [np.int64(0), np.int64(1), np.int64(2)]
  sentiment nulls: 0
  Size: 43.1 MB

test_set.csv:
  Colonnes: ['Id', 'ProductId', 'UserId', 'Time', 'Summary', 'Text', 'cleaned', 'sentiment']
  sentiment unique: [np.int64(0), np.int64(1), np.int64(2)]
  sentiment nulls: 0
  Size: 43.4 MB

✅ Done!
